In [18]:
import sys
from pathlib import Path

# go to: gnn_drone_project/residual_correction
ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd().resolve().parent

sys.path.append(str(ROOT))

In [19]:


import torch
import numpy as np
from torch_geometric.data import Data

from models import build_model
from inference.inference_engine import InferenceEngine


In [20]:

# =========================================================
# CONFIG
# =========================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_PATH = ROOT / "checkpoints" / "best_nnconv.pt"
MODEL_NAME = "nnconv"

IN_DIM = 73  # MUST match training exactly

# =========================================================
# LOAD MODEL
# =========================================================
model = build_model(MODEL_NAME, in_dim=IN_DIM)

state = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state)

engine = InferenceEngine(model, DEVICE)

print("✅ Model loaded OK")

# =========================================================
# BUILD REALISTIC GRAPH TEST
# =========================================================
N = 5  # drones

x = torch.randn((N, IN_DIM), dtype=torch.float32)
pos = torch.randn((N, 3), dtype=torch.float32)

edge_index = torch.tensor([
    [0, 1, 2, 3],
    [1, 2, 3, 4]
], dtype=torch.long)

edge_attr = torch.randn((4, 7), dtype=torch.float32)

graph = Data(
    x=x,
    pos=pos,
    edge_index=edge_index,
    edge_attr=edge_attr
)

# =========================================================
# RUN INFERENCE
# =========================================================
with torch.no_grad():
    out = engine.predict(graph)

# =========================================================
# SAFETY CHECKS
# =========================================================
assert out.shape == (N, 3), f"Wrong output shape: {out.shape}"
assert torch.isfinite(out).all(), "NaN or Inf detected in output"

print("✅ Output shape:", out.shape)
print("✅ Output sample:", out[0])
print("🎉 Inference pipeline is VALID")

✅ Model loaded OK
✅ Output shape: torch.Size([5, 3])
✅ Output sample: tensor([-1.9110e-01, -7.5046e-02,  9.5976e-08])
🎉 Inference pipeline is VALID
